In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from depth_anything_3.api import DepthAnything3
from depth_anything_3.utils.alignment import compute_sky_mask

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_window_contents
)
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)
from src.common.geometry.depth import transform_cam_to_ego
from src.common.geometry.transform import make_transform, invert_transform
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.depth import plot_depth_with_original_image, plot_pseudo_lidar_with_ground_truth
from src.common.frame_ops import create_sliding_windows

from src.depth_anything3.inference import get_metric_depth, get_pseudo_lidar

# Resolve paths relative to this notebook directory
MODEL_NAME = "DA3METRIC-LARGE"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = DepthAnything3.from_pretrained(f"depth-anything/{MODEL_NAME}")
model = model.to(device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and camera channel
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"
LIDAR_CHANNEL = "LIDAR_TOP"

# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    sample_annotations_all, instances_all)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]

# Show the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = sample_contents_cam["sample_data"]
    sample_contents_lidar = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                                sensor_token=sensor_lookup[LIDAR_CHANNEL])
    sample_data_lidar = sample_contents_lidar["sample_data"]
    calibrated_sensor_lidar = sample_contents_lidar["calibrated_sensor"]
    ego_pose_lidar = sample_contents_lidar["ego_pose"]
    # Read and display the image
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    # Read the point cloud
    lidar_path = NUSCENES_ROOT / sample_data_lidar["filename"]
    lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
    points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
    intensity = lidar_points[:, 3]  # Extract intensity values
    # Transform the points from the LiDAR frame to the global frame
    lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                              lidar_translation=calibrated_sensor_lidar["translation"],
                                              lidar_quaternion=calibrated_sensor_lidar["rotation"])
    lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                  ego_translation=ego_pose_lidar["translation"],
                                                  ego_quaternion=ego_pose_lidar["rotation"])
    # Visualize the point cloud
    fig = plot_pointcloud(lidar_points_global,
                          axis_translation=ego_pose_lidar["translation"],
                          axis_quaternion=ego_pose_lidar["rotation"])
    fig.show()
    

In [ ]:
# Depth estimation by DepthAnything3
# Show the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                                sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = sample_contents_cam["sample_data"]
    calibrated_sensor_cam = sample_contents_cam["calibrated_sensor"]
    ego_pose_cam = sample_contents_cam["ego_pose"]
    sample_contents_lidar = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                                sensor_token=sensor_lookup[LIDAR_CHANNEL])
    sample_data_lidar = sample_contents_lidar["sample_data"]
    calibrated_sensor_lidar = sample_contents_lidar["calibrated_sensor"]
    ego_pose_lidar = sample_contents_lidar["ego_pose"]
    # Read the image
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)

    # Inference depth using DepthAnything3 (without pose conditioning)
    prediction = model.inference([image])
    metric_depths, scaled_intrinsics = get_metric_depth(prediction,
                                                        model_name=MODEL_NAME,
                                                        camera_intrinsics=[calibrated_sensor_cam["camera_intrinsic"]],
                                                        original_image_width=image.width,
                                                        original_image_height=image.height)
    # Get the sky mask from the depth prediction if available
    non_sky_mask = compute_sky_mask(prediction.sky) if prediction.sky is not None else None
    # Show the depth map
    plot_depth_with_original_image(metric_depths[0], np.array(image))
    # Generate a point cloud by pseudo-lidar from the depth map and transform to global coordinates
    pseudo_lidar_points = get_pseudo_lidar(metric_depths[0],
                                           np.array(scaled_intrinsics[0]),
                                           non_sky_mask=non_sky_mask)
    pseudo_points_ego = transform_cam_to_ego(pseudo_lidar_points,
                                             camera_translation=calibrated_sensor_cam["translation"],
                                             camera_quaternion=calibrated_sensor_cam["rotation"])
    pseudo_points_global = transform_ego_to_global(pseudo_points_ego,
                                                   ego_translation=ego_pose_cam["translation"],
                                                   ego_quaternion=ego_pose_cam["rotation"])

    ###### Ground truth LiDAR comparison ######
    # Read the point cloud
    lidar_path = NUSCENES_ROOT / sample_data_lidar["filename"]
    lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
    points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
    intensity = lidar_points[:, 3]  # Extract intensity values
    # Transform the points from the LiDAR frame to the global frame
    lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                                lidar_translation=calibrated_sensor_lidar["translation"],
                                                lidar_quaternion=calibrated_sensor_lidar["rotation"])
    lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                    ego_translation=ego_pose_lidar["translation"],
                                                    ego_quaternion=ego_pose_lidar["rotation"])

    # Visualize the point cloud using Open3D
    fig = plot_pseudo_lidar_with_ground_truth(
        pseudo_lidar_points=pseudo_points_global,
        ground_truth_points=lidar_points_global,
        axis_translation=ego_pose_lidar["translation"],
        axis_quaternion=ego_pose_lidar["rotation"]
    )
    fig.show()

In [ ]:
# Pose-conditioned depth estimation with sliding window
MODEL_NAME = "DA3NESTED-GIANT-LARGE-1.1"
WINDOW_SIZE = 7  # Number of frames in the sliding window
STRIDE = 3  # Stride for sliding window
model = DepthAnything3.from_pretrained(f"depth-anything/{MODEL_NAME}")
model = model.to(device=device)

# Create a sliding windows
window_ranges, used_ranges = create_sliding_windows(len(samples), window_size=WINDOW_SIZE, stride=STRIDE)

for window_count, ((i_start, i_end), (u_start, u_end)) in enumerate(zip(window_ranges, used_ranges)):
    # Get the contents of the current sliding window
    window_contents_cam = get_sample_window_contents(i_start, i_end-1, samples, sample_data, ego_poses, calibrated_sensors,
                                                     sensor_token=sensor_lookup[CAMERA_CHANNEL])
    
    window_samples = window_contents_cam["samples"]
    window_sample_data_cam = window_contents_cam["sample_data"]
    window_ego_poses_cam = window_contents_cam["ego_poses"]
    window_calibrated_sensors_cam = window_contents_cam["calibrated_sensors"]

    # Get the images and camera intrinsics for the sliding window
    images = [Image.open(NUSCENES_ROOT / sd["filename"]) for sd in window_sample_data_cam.values()]
    intrinsics = []
    extrinsics = []

    # Calculate the extrinsics for the sliding window
    for sd in window_sample_data_cam.values():
        cs = window_calibrated_sensors_cam[sd["calibrated_sensor_token"]]
        intrinsics.append(cs["camera_intrinsic"])
        ep = window_ego_poses_cam[sd["ego_pose_token"]]
        camera_to_ego = make_transform(quaternion=cs["rotation"], translation=cs["translation"])
        ego_to_global = make_transform(quaternion=ep["rotation"], translation=ep["translation"])
        camera_to_global = ego_to_global @ camera_to_ego
        global_to_camera = invert_transform(camera_to_global)
        extrinsics.append(global_to_camera)
    
    # Stack intrinsics and extrinsics for the sliding window
    intrinsics = np.stack(intrinsics, axis=0).astype(np.float32)
    extrinsics = np.stack(extrinsics, axis=0).astype(np.float32)

    # Inference depth using DepthAnything3 with pose conditioning
    prediction = model.inference(images, extrinsics=extrinsics, intrinsics=intrinsics)
    metric_depths, scaled_intrinsics = get_metric_depth(prediction, model_name=MODEL_NAME)
    
    for used_idx in range(u_start, u_end):
        used_image = images[used_idx]
        used_metric_depth = metric_depths[used_idx]
        used_scaled_intrinsic = scaled_intrinsics[used_idx]
        used_sample_data = list(window_sample_data_cam.values())[used_idx]
        used_calibrated_sensor_cam = window_calibrated_sensors_cam[used_sample_data["calibrated_sensor_token"]]
        used_ego_pose_cam = window_ego_poses_cam[used_sample_data["ego_pose_token"]]
        
        # Get the sky mask from the depth prediction if available (NestedDepthAnything3Net.forward has doesn't copy the sky mask to output.sky. TODO: It needs to be added in the model repo and pull request)
        non_sky_mask = compute_sky_mask(prediction.sky[used_idx]) if prediction.sky is not None else None
        # Generate a point cloud by pseudo-lidar from the depth map and transform to global coordinates
        pseudo_lidar_points = get_pseudo_lidar(used_metric_depth,
                                               np.array(used_scaled_intrinsic),
                                               non_sky_mask=non_sky_mask)
        pseudo_points_ego = transform_cam_to_ego(pseudo_lidar_points,
                                                 camera_translation=used_calibrated_sensor_cam["translation"],
                                                 camera_quaternion=used_calibrated_sensor_cam["rotation"])
        pseudo_points_global = transform_ego_to_global(pseudo_points_ego,
                                                       ego_translation=used_ego_pose_cam["translation"],
                                                       ego_quaternion=used_ego_pose_cam["rotation"])

        # Show the depth map and pseudo LiDAR in the first sliding window
        if window_count == 0:
            plot_depth_with_original_image(used_metric_depth, np.array(used_image))

            ###### Ground truth LiDAR comparison ######
            sample_contents_lidar = get_sample_contents(i_start+used_idx, samples, sample_data, ego_poses, calibrated_sensors,
                                                        sensor_token=sensor_lookup[LIDAR_CHANNEL])
            # Read the point cloud
            lidar_path = NUSCENES_ROOT / sample_contents_lidar["sample_data"]["filename"]
            lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
            points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
            intensity = lidar_points[:, 3]  # Extract intensity values
            # Transform the points from the LiDAR frame to the global frame
            calibrated_sensor_lidar = sample_contents_lidar["calibrated_sensor"]
            lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                                      lidar_translation=calibrated_sensor_lidar["translation"],
                                                      lidar_quaternion=calibrated_sensor_lidar["rotation"])
            ego_pose_lidar = sample_contents_lidar["ego_pose"]
            lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                          ego_translation=ego_pose_lidar["translation"],
                                                          ego_quaternion=ego_pose_lidar["rotation"])
        
            # Visualize the point cloud using Open3D
            fig = plot_pseudo_lidar_with_ground_truth(
                pseudo_lidar_points=pseudo_points_global,
                ground_truth_points=lidar_points_global,
                axis_translation=ego_pose_lidar["translation"],
                axis_quaternion=ego_pose_lidar["rotation"]
            )
            fig.show()

